# Alytes-ReID — Setup & Training

**Who runs this notebook**: researcher / developer (once per project, or when new labeled data arrives).

**What it does**:
1. Install dependencies
2. Enter API keys (Roboflow, optional iNaturalist)
3. Download training data
4. Train YOLO11 detection model
5. Validate SAM2 segmentation
6. Train Re-ID model (when labeled data is available)
7. Save models to Google Drive → used by `02_toad_reid.ipynb`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/claude/alytes-reid-system-qgved/notebooks/01_setup_and_training.ipynb)

---
## 0. Inserisci la tua API Key

Esegui questa cella: comparirà un campo di testo dove incollare la chiave.

**Come ottenere la chiave Roboflow** (gratis):
1. Vai su https://roboflow.com e crea un account
2. Clicca sulla tua icona in alto a destra → **Settings**
3. Nella sezione **API Keys** copia la chiave
4. Incollala nel campo che appare qui sotto

> Le chiavi restano solo in questa sessione di Colab e non vengono mai salvate su disco.

In [ ]:
print("=" * 50)
print("  INSERISCI LA TUA API KEY DI ROBOFLOW")
print("=" * 50)
print()
print("Incolla la chiave nel campo qui sotto e premi Invio.")
print("(Se non ce l'hai, lascia vuoto e premi Invio per saltare)")
print()

ROBOFLOW_API_KEY = input("Roboflow API Key: ")

if ROBOFLOW_API_KEY:
    print()
    print("Chiave Roboflow inserita correttamente!")
else:
    print()
    print("Nessuna chiave inserita. Il dataset Roboflow verra' saltato.")
    print("Potrai usare solo le immagini da iNaturalist.")

print()
print("-" * 50)
print("Opzionale: token HuggingFace (per modelli SAM2)")
print("(Se non sai cos'e', lascia vuoto e premi Invio)")
print()

HF_TOKEN = input("HuggingFace Token (opzionale): ")

if HF_TOKEN:
    print("Token HuggingFace inserito.")
else:
    print("Nessun token HF. Verra' usata la versione base di SAM2.")

print()
print("Tutto pronto! Passa alla cella successiva.")

---
## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

# Clone repo
!git clone -b claude/alytes-reid-system-qgved https://github.com/danort92/Alytes-ReID.git 2>/dev/null || (cd Alytes-ReID && git pull)
%cd Alytes-ReID

# Install dependencies
!pip install -r requirements.txt -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'
!mkdir -p {DRIVE_MODEL_DIR}

print('\nSetup complete!')

---
## 2. Data Download

In [ ]:
from pathlib import Path

# --- 2a. iNaturalist (no key needed) ---
from src.data.download_inat import download_alytes_images

MAX_INAT_IMAGES = 500  # increase to 1000 if you have time

inat_images = download_alytes_images(
    output_dir=Path('data/raw/inaturalist'),
    max_images=MAX_INAT_IMAGES,
)
print(f'iNaturalist: {len(inat_images)} images downloaded')

In [ ]:
# --- 2b. Roboflow frogs dataset (pre-annotated, requires API key) ---
if ROBOFLOW_API_KEY:
    !pip install roboflow -q
    from src.data.download_roboflow import download_roboflow_dataset

    roboflow_dir = download_roboflow_dataset(
        output_dir=Path('data/raw/roboflow'),
        api_key=ROBOFLOW_API_KEY,
        dataset_format='yolov8',
    )
    print(f'Roboflow dataset downloaded to: {roboflow_dir}')
else:
    print('Roboflow key not provided — skipping pre-annotated dataset.')
    print('You can annotate images manually using Label Studio or Roboflow.')

---
## 3. Data Exploration

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

sample = random.sample(inat_images, min(8, len(inat_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample iNaturalist Images (Alytes obstetricans)')
plt.tight_layout()
plt.show()

print(f'Total images available: {len(inat_images)}')

---
## 4. Dataset Preparation

Organizes images and labels into YOLO format with train/val/test splits.

In [ ]:
from src.data.prepare_dataset import split_dataset, create_yolo_dataset_yaml

# Use Roboflow data if available (already annotated), otherwise use iNat
if ROBOFLOW_API_KEY and Path('data/raw/roboflow').exists():
    images_source = Path('data/raw/roboflow/train/images')
    labels_source = Path('data/raw/roboflow/train/labels')
    print('Using Roboflow annotated data')
else:
    # For iNaturalist images, manual annotation is required first.
    # Recommended: upload to Roboflow for annotation, then export.
    print('⚠ iNaturalist images need bounding box annotation before training.')
    print('  Option 1: Annotate with Roboflow (free): https://roboflow.com')
    print('  Option 2: Annotate with Label Studio: https://labelstud.io')
    print('  Then run this cell again with annotated data.')
    images_source = None

if images_source and images_source.exists():
    split_dataset(
        images_dir=images_source,
        labels_dir=labels_source,
        output_dir=Path('data/processed/detection'),
    )
    dataset_yaml = create_yolo_dataset_yaml(
        dataset_dir=Path('data/processed/detection'),
        classes=['toad'],
    )
    print(f'Dataset ready: {dataset_yaml}')

---
## 5. YOLO11 Detection Training

YOLO11 is the latest Ultralytics model (2024). Weights download automatically.

In [ ]:
# Configure training
# Edit these values or modify config/detection.yaml directly

YOLO_MODEL = 'yolo11s'  # yolo11n (fastest) | yolo11s | yolo11m | yolo11l | yolo11x
EPOCHS = 100
BATCH_SIZE = 16  # reduce to 8 if GPU OOM

print(f'Model: {YOLO_MODEL}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}')

In [ ]:
from src.detection.train import load_config, train_detector

config = load_config(Path('config/detection.yaml'))

# Override config with the values set above
config['model']['architecture'] = YOLO_MODEL
config['training']['epochs'] = EPOCHS
config['training']['batch_size'] = BATCH_SIZE

# Train
best_weights = train_detector(config)
print(f'\nBest weights: {best_weights}')

---
## 6. Evaluate Detection

In [ ]:
from src.detection.evaluate import evaluate_model

metrics = evaluate_model(best_weights, config)

print('\n=== Detection Results ===')
for name, value in metrics.items():
    print(f'  {name:15s}: {value:.4f}')

---
## 7. SAM2 Segmentation Validation

In [ ]:
# Install SAM2 (not in requirements.txt to keep Colab install light)
!pip install segment-anything-2 -q

if HF_TOKEN:
    import os
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HuggingFace token set.')

In [ ]:
from src.segmentation.segment import ToadSegmenter
from src.detection.predict import load_detector, detect_toads, get_best_detection
from src.utils.visualization import draw_detections, draw_mask_overlay

detector = load_detector(best_weights)
segmenter = ToadSegmenter()  # loads SAM2 lazily on first call

# Pick a test image
test_path = inat_images[0]
test_img = cv2.cvtColor(cv2.imread(str(test_path)), cv2.COLOR_BGR2RGB)

detections = detect_toads(detector, test_path)
best_det = get_best_detection(detections)

if best_det:
    cropped, mask = segmenter.segment_and_crop(test_img, best_det['bbox'])
    overlay = draw_mask_overlay(
        draw_detections(test_img, detections),
        segmenter.segment_from_bbox(test_img, best_det['bbox'])
    )
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(test_img); axes[0].set_title('Original')
    axes[1].imshow(overlay); axes[1].set_title('Detection + Mask')
    axes[2].imshow(cropped); axes[2].set_title('Cropped')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No toad detected in test image.')

---
## 8. Re-ID Model Training

Requires labeled data organized as `data/processed/reid/<INDIVIDUAL_ID>/<image>.png`.  
Upload the biologist's photos and run the preparation script, then train.

In [ ]:
# Upload labeled re-ID data from local machine
# from google.colab import files
# uploaded = files.upload()  # upload a zip archive
# !unzip -q your_labeled_data.zip -d data/processed/reid/

# Or copy from Drive:
# !cp -r '/content/drive/MyDrive/Alytes-ReID/reid_data' data/processed/reid/

In [ ]:
# Train Re-ID model
# from src.reid.train import train_reid
#
# reid_config = load_config(Path('config/reid.yaml'))
# reid_model_path = train_reid(reid_config, data_dir=Path('data/processed/reid'))
# print(f'Re-ID model saved to: {reid_model_path}')

---
## 9. Save Models to Google Drive

In [ ]:
import shutil

# Detection model
shutil.copy2(str(best_weights), f'{DRIVE_MODEL_DIR}/detection_best.pt')
print(f'Detection model saved to Drive: {DRIVE_MODEL_DIR}/detection_best.pt')

# Re-ID model (uncomment after training)
# shutil.copy2(str(reid_model_path), f'{DRIVE_MODEL_DIR}/reid_model.pt')

# Re-ID database (uncomment after building)
# shutil.copytree('data/models/reid', f'{DRIVE_MODEL_DIR}/reid_db', dirs_exist_ok=True)

print('Models saved to Google Drive. Ready to use in 02_toad_reid.ipynb')